In [ ]:
!pip install transformers torch torchvision sentencepiece gradio language-tool-python gtts pillow

In [ ]:
import gradio as gr
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import language_tool_python
from gtts import gTTS
import tempfile

# Load Microsoft's Handwriting OCR Model
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Grammar correction
tool = language_tool_python.LanguageTool('en-US')

In [ ]:
def extract_text(image):
    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    generated_ids = model.generate(pixel_values)

    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return generated_text

In [ ]:
def correct_text(raw_text):
    if not raw_text:
        return ""

    matches = tool.check(raw_text)

    corrected = language_tool_python.utils.correct(raw_text, matches)

    return corrected

In [ ]:
def text_to_speech(text):
    if not text:
        return None

    tts = gTTS(text=text, lang='en')

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")

    tts.save(tmp.name)

    return tmp.name

In [ ]:
def process_image(image):
    raw = extract_text(image)

    corrected = correct_text(raw)

    audio = text_to_speech(corrected)

    return raw, corrected, audio

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 📝 AI Handwriting to Speech")

    image_input = gr.Image(type="pil")

    btn = gr.Button("Convert")

    raw_output = gr.Textbox(label="Extracted Text")

    corrected_output = gr.Textbox(label="Corrected Text")

    audio_output = gr.Audio()

    btn.click(
        fn=process_image,
        inputs=image_input,
        outputs=[raw_output, corrected_output, audio_output]
    )

demo.launch(debug=True)

In [ ]:
!pip install transformers torch torchvision sentencepiece jiwer opencv-python pillow

In [ ]:
import torch
from PIL import Image
import cv2
import numpy as np

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# LOAD MODEL (THIS WAS MISSING)
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

In [ ]:
def preprocess_for_trocr(pil_image):
    img = np.array(pil_image)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.convertScaleAbs(gray, alpha=2.5, beta=30)

    thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)[1]

    coords = cv2.findNonZero(thresh)

    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        cropped = img[y:y+h, x:x+w]
    else:
        cropped = img

    cropped = cv2.resize(cropped, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)

    return Image.fromarray(cropped)

In [ ]:
def split_lines(pil_image):
    import cv2
    import numpy as np

    img = np.array(pil_image)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)[1]

    horizontal_sum = np.sum(thresh, axis=1)

    lines = []
    start = None

    for i, val in enumerate(horizontal_sum):
        if val > 0 and start is None:
            start = i
        elif val == 0 and start is not None:
            end = i
            if end - start > 10:
                line_img = img[start:end, :]
                lines.append(line_img)
            start = None

    return lines


In [ ]:
def extract_text(image):
    lines = split_lines(image)

    full_text = ""

    # 🔥 fallback if no lines detected
    if len(lines) == 0:
        lines = [np.array(image)]

    for line in lines:
        line_img = Image.fromarray(line)

        processed = preprocess_for_trocr(line_img)

        pixel_values = processor(images=processed, return_tensors="pt").pixel_values

        generated_ids = model.generate(
            pixel_values,
            max_length=100,
            num_beams=4
        )

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        full_text += text + " "

    return full_text.strip()

In [ ]:
def load_dataset(label_file, image_folder):
    dataset = []

    with open(label_file, "r") as f:
        for line in f:
            if "|" in line:
                img_name, text = line.strip().split("|", 1)
            else:
                parts = line.strip().split(" ", 1)
                if len(parts) < 2:
                    continue
                img_name, text = parts

            dataset.append({
                "image": f"{image_folder}/images/{img_name}",  # IMPORTANT FIX
                "truth": text
            })

    return dataset

In [ ]:
from jiwer import wer, cer

def evaluate_model(dataset):
    total_wer = 0
    total_cer = 0

    for data in dataset:
        img = Image.open(data["image"])
        pred = extract_text(img)
        truth = data["truth"]

        w = wer(truth, pred)
        c = cer(truth, pred)

        total_wer += w
        total_cer += c

        print("\n------------------------")
        print(f"Truth: {truth}")
        print(f"Pred : {pred}")
        print(f"WER  : {w:.2f}")
        print(f"CER  : {c:.2f}")

    avg_wer = total_wer / len(dataset)
    accuracy = (1 - avg_wer) * 100

    print("\n========================")
    print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset = load_dataset(
    "/content/drive/MyDrive/dataset/labels.txt",
    "/content/drive/MyDrive/dataset"
)

In [ ]:
evaluate_model(dataset[:10])

In [ ]:
!pip install gtts
import gradio as gr
from gtts import gTTS
import tempfile

# 🔊 Text to Speech
def text_to_speech(text):
    if not text:
        return None
    tts = gTTS(text=text, lang='en')
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tts.save(tmp.name)
    return tmp.name

# 🔥 Main pipeline
def run_pipeline(image):
    text = extract_text(image)
    audio = text_to_speech(text)
    return text, audio

# 🎨 Stylish UI
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="purple"
    )
) as demo:

    # 🌈 Gradient Title
    gr.Markdown(
        """
        <h1 style='text-align: center;
                   background: linear-gradient(to right, #4facfe, #00f2fe);
                   -webkit-background-clip: text;
                   color: transparent;'>
            📝 AI Handwritten notes to speech conversion System for Dyslexic students
        </h1>
        <p style='text-align: center; font-size:16px;'>
            Upload handwritten text → AI extracts → Converts to speech
        </p>
        """
    )

    with gr.Row():

        # 📸 Left side
        with gr.Column():
            image_input = gr.Image(type="pil", label="📸 Upload Handwritten Image")

            btn = gr.Button("🚀 Process", variant="primary")

        # 🧠 Right side
        with gr.Column():
            output_text = gr.Textbox(
                label="🧠 Extracted Text",
                lines=5,
                placeholder="AI output will appear here..."
            )

            output_audio = gr.Audio(label="🔊 Speech Output")

    btn.click(
        fn=run_pipeline,
        inputs=image_input,
        outputs=[output_text, output_audio]
    )

    # Footer
    gr.Markdown(
        """
        <hr>
        <p style='text-align:center;'>
            Built with ❤️ using TrOCR + NLP + TTS
        </p>
        """
    )

demo.launch()